In [1]:
from pymol import cmd
cmd.feedback("disable", "all", "everything")  # Disables all output, comment this if something doesn't go as expected
import pandas as pd
import os

In [2]:
# ALL_RESULTS_TRIPLETS = pd.read_csv("../processed/protein_priorization/triplets.tsv", sep='\t')
# ALL_RESULTS_TRIPLETS[(ALL_RESULTS_TRIPLETS['RMSD_A'] < 2) & (ALL_RESULTS_TRIPLETS['RMSD_B'] < 2) & (ALL_RESULTS_TRIPLETS['RMSD_C'] < 2)]

In [49]:
ALL_RESULTS_PAIRS = pd.read_csv("../processed/protein_priorization/pairs.tsv", sep='\t')
ALL_RESULTS_PAIRS[ALL_RESULTS_PAIRS['Protein RMSD'] < 2]

,Protein1,Pocket1,InterPro-Pocket1,Protein2,Pocket2,InterPro-Pocket2,PocketVec distance,Protein RMSD,Protein SEQ ID (CO),Protein SEQ ID (NW)
3,P9WFU5,alphafold3_P9WFU5_model_2_pocket_1,Catalytic Domain (ATP Binding Site);Other too ...,P9WFV1,alphafold2_P9WFV1_model_0_pocket_4,Catalytic Domain (ATP Binding Site),0.121,1.66,32.9167,26.00
4,P9WFV7,swissmodel_P9WFV7_model_1_pocket_2,Anticodon Binding Domain,P9WFW5,chai1_P9WFW5_model_4_pocket_2,Catalytic Domain (ATP Binding Site);Other too ...,0.121,1.92,25.0000,12.75
5,P9WFW3,alphafold3_P9WFW3_model_0_pocket_2,Other too broad/unspecified functional entities,P9WQA1,swissmodel_P9WQA1_model_0_pocket_2,Other too broad/unspecified functional entities,0.122,1.66,25.2654,12.58
9,P9WFW3,alphafold3_P9WFW3_model_0_pocket_2,Other too broad/unspecified functional entities,P9WQA1,alphafold2_P9WQA1_model_0_pocket_2,Other too broad/unspecified functional entities,0.125,1.66,25.2654,12.58
35,P9WFW3,alphafold2_P9WFW3_model_0_pocket_2,NaN,P9WQA1,alphafold2_P9WQA1_model_0_pocket_2,Other too broad/unspecified functional entities,0.128,1.66,25.2654,12.58
...,...,...,...,...,...,...,...,...,...,...
35495,P9WFT5,chai1_P9WFT5_model_3_pocket_2,Anticodon Binding Domain;Catalytic Domain (ATP...,P9WFT7,alphafold3_P9WFT7_model_2_pocket_2,Other too broad/unspecified functional entities,0.314,1.91,29.3827,21.25
35497,P9WFT5,alphafold3_P9WFT5_model_1_pocket_2,Anticodon Binding Domain;Catalytic Domain (ATP...,P9WFT7,alphafold3_P9WFT7_model_2_pocket_2,Other too broad/unspecified functional entities,0.314,1.91,29.3827,21.25
35526,P9WFU9,alphafold2_P9WFU9_model_0_pocket_1,Other too broad/unspecified functional entities,P9WFW1,swissmodel_P9WFW1_model_0_pocket_3,Catalytic Domain (ATP Binding Site),0.322,0.63,24.7059,9.68
35538,P9WFT5,alphafold3_P9WFT5_model_1_pocket_2,Anticodon Binding Domain;Catalytic Domain (ATP...,P9WFT7,chai1_P9WFT7_model_4_pocket_2,Other too broad/unspecified functional entities,0.324,1.91,29.3827,21.25


In [45]:
def prepare_pymol_session(pocket1, pocket2, outdir):

    st1 = "_".join(pocket1.split("_")[:4])
    p1 = "_".join(pocket1.split("_")[4:])
    st2 = "_".join(pocket2.split("_")[:4])
    p2 = "_".join(pocket1.split("_")[4:])

    PATH_TO_STRUCTURES = "../processed/aligned_relaxed_structures/"
    PATH_TO_POCKETS = "../processed/detected_pockets/"


    # Define some colors
    COLORS = ['wheat', 'grey', 'skyblue']

    # Initialize PyMOL
    cmd.reinitialize()

    # Prettify session 
    cmd.do("set orthoscopic, on")
    cmd.do("set ray_trace_fog, 0")
    cmd.do("set depth_cue, 0")
    cmd.do("set antialias, 4")
    cmd.do("set ray_trace_mode, 1")
    cmd.do("set ray_trace_gain, 0.005")
    cmd.do("bg_color white")
    cmd.do("set spec_reflect, 0")
    cmd.do("set transparency, 0.1")  
    cmd.do("set sphere_scale, 2")
    cmd.do("set internal_gui_width, 400")

    PYMOL_OBJECTS = []

    # Load all structures
    for c, (st, p) in enumerate(zip([st1, st2], [p1, p2])):

        uni = st.split("_")[1]

        # Load structure
        cmd.load(os.path.join(PATH_TO_STRUCTURES, uni, f"{st}.pdb"), st)
        cmd.color(COLORS[c], st)
        cmd.show("cartoon", st)

        # Load pocket
        cmd.load(os.path.join(PATH_TO_POCKETS, uni, st, "pockets", f"{p}.pdb"), f"{st}_{p}")
        cmd.color(COLORS[c], f"{st}_{p}")
        cmd.show("spheres", f"{st}_{p}")

        # Copy structure to pocket and remove structure
        cmd.create(f"{st}_{p}", f"{st}_{p} or {st}")
        cmd.delete(st)

        # Append pymol object
        PYMOL_OBJECTS.append(f"{st}_{p}")

    # Align all to reference
    ref = PYMOL_OBJECTS[0]
    for al in PYMOL_OBJECTS[1:]:
        cmd.super(f"{al} and name CA", f"{ref} and name CA")

    # Save PyMOL session
    cmd.reset()
    cmd.save(os.path.join(outdir, f"{pocket1}__{pocket2}.pse"))

In [46]:
pocket1 = "alphafold3_P9WFU5_model_2_pocket_1"
pocket2 = "alphafold2_P9WFV1_model_0_pocket_4"
outdir = "/home/acomajuncosa/Documents/tmp"

prepare_pymol_session(pocket1, pocket2, outdir)